# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.71 GB
MemFree: 165.22 GB
MemAvailable: 890.42 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################



## 2. SEML Pipeline

In [3]:
from src.data.FKTC_datasets import load_dataset_from_name
from src.reliability.response_generator import ResponseGenerator
from src.models import base_models
import torch

import pandas as pd
import numpy as np
from sklearn import metrics

def run_reliability_eval(
    model_name,
    max_new_tokens,
    temperature,
    use_beam_search,
    strategy,
    dataset_name,
    device='cuda',
    seed=123,
    n_repeats=5,
    n_beams=5,
    cache_path=CACHE_PATH
    ):
    # LOAD DATASET
    qa_dataset = load_dataset_from_name(dataset_name, max_entries=100)

    # INITIALIZE RESULTS LIST
    results = []

    n_steps = 0
    total_steps = len(qa_dataset) * (n_repeats if not use_beam_search else 1)
    generator = ResponseGenerator(base_models[model_name])
    for query_idx, (query, true_answer) in enumerate(qa_dataset):
        run_results = generator.generate_response(query, strategy, true_answer, max_new_tokens, temperature, use_beam_search, n_repeats=n_repeats, n_beams=n_beams)
        for result_dict in run_results:
            print(f"  TOTAL: {n_steps + 1}/{total_steps}, MODEL: {model_name}, QUERY: {query_idx}, STRATEGY: {strategy}, MAX_NEW_TOKENS: {max_new_tokens}, RUN: {result_dict['run']}/{n_repeats}")
            # Store the results in the list
            results.append({
                "Query": query,
                "Run": result_dict['run'],
                "Generated Response": result_dict['output_text'],
                "P": result_dict['beam_prob'],
                "P_adj": result_dict['beam_prob_adj'],
                "Entropy": result_dict['entropy'],
                "Is Correct": result_dict['is_correct'],
                "Token Probabilities": result_dict['token_probs']
            })
            print(f"    IS_CORRECT: {result_dict['is_correct']}, PROB: {result_dict['beam_prob']:.2f}, ADJ_PROB: {result_dict['beam_prob_adj']:.2f}, ENTROPY: {result_dict['entropy']:.2f}")
            n_steps += 1
    
    # Generate custom file name based on parameters
    beam_search_str = "beam" if use_beam_search else "sample"
    strategy_str = strategy.replace(" ", "_").lower()  # Replace spaces with underscores for file names
    file_base = f"{model_name}_{dataset_name}_{beam_search_str}_{max_new_tokens}_tokens_{temperature}_temp_{strategy_str}"

    # Optional: Create a directory for saving the results if not already existing
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    save_dir = os.path.join(results_path, "reliability_eval")
    os.makedirs(save_dir, exist_ok=True)

    # Generate file paths
    raw_table_path = os.path.join(save_dir, f"{file_base}_raw_table.xlsx")
    scores_table_path = os.path.join(save_dir, f"{file_base}_scores.xlsx")
    
    df_results = pd.DataFrame(results)
    
    # Calculate P_sem as the proportion of True values in 'Is Correct' per group
    df_results['P_sem'] = df_results.groupby(['Query'])['Is Correct'].transform('mean')

    # Define custom AUC calculation
    def custom_auc_roc(corrects, scores):
        fpr, tpr, thresholds = metrics.roc_curve(corrects, scores)
        return metrics.auc(fpr, tpr)
    
    def calculate_scores(df):
        y_true = df['Is Correct'].values

        # Calculate various AUCROC and AUCPR scores
        scores_dict = {}
        metrics_to_calculate = {
            'sample': 'P',
            'adj': 'P_adj',
            'entr': 'Entropy',
            'sem': 'P_sem'
        }

        for key, score_column in metrics_to_calculate.items():
            y_scores = df[score_column].values
            if len(set(y_true)) > 1:  # Ensure at least two classes are present
                aucroc = custom_auc_roc(y_true, y_scores)
            else:
                aucroc = np.nan
            aucpr = metrics.average_precision_score(y_true, y_scores)
            accuracy = np.mean(y_true)

            scores_dict[f'AUCROC_{key}'] = aucroc
            scores_dict[f'AUCPR_{key}'] = aucpr
        
        scores_dict['Accuracy'] = accuracy
    
        return pd.Series(scores_dict)

    # Apply the calculate_scores function to the entire DataFrame
    df_scores = calculate_scores(df_results)

    # Convert df_scores to a DataFrame with a single row for consistent saving format
    df_scores = df_scores.to_frame().T

    # Save the original detailed results to an Excel file
    df_results.to_excel(raw_table_path, index=False)
    df_scores.to_excel(scores_table_path, index=False)

    return {"results": df_results, "scores": df_scores}

In [4]:
import itertools

# Fixed parameters
fixed_params = {
    'device': 'cuda',
    'seed': 123,
    'n_repeats': 10,
    'n_beams': 5,
    'cache_path': CACHE_PATH
}

# Grid parameters
grid_params = {
    'model_name': [
        'Llama-3-8B',
        'TinyLlama-Chat',
        'Bloomz',
        'GPT2-Large',
        'TinyLlama',
        # 'Llama-3-8B-AWQ-4bit'  # Uncomment if needed
    ],
    'max_new_tokens': [
        10,
        15,
        20,
        30,
        40
    ],
    'temperature': [
        0.1
    ],
    'use_beam_search': [
        True,
        False
    ],
    'strategy': [
        "Fact Statement",
        "Completion",
        "Definitive Statement",
        "Fill-in-the-Blank",
        "Structured Answer Prompt",
        "Direct Instruction",
        "Contextual Prompts",
        "Question-Answer Pairs",
        "Direct Answer",
        "Q&A Format",
        "Instructional",
        "Summary",
        "True Completion",
        "Direct Completion",
        "Answer Completion",
        "Direct Query",
        "Yes/No Confirmation",
        "Follow-Up Inquiry",
        "Factual Retrieval",
        "First Thought",
        "Deductive Reasoning"
    ],
    'dataset_name': [
        'toy-qa-dataset',
        'P101',
        'P103',
        'P108',
        'P127',
        'P1376',
        'P1412',
        'P159',
        'P17',
        'P176',
        'P178',
        'P19',
        'P20',
        'P264',
        'P27',
        'P276',
        'P30',
        'P364',
        'P37',
        'P495',
        'P740'
    ]
}

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['model_name'],
    grid_params['max_new_tokens'],
    grid_params['temperature'],
    grid_params['use_beam_search'],
    grid_params['strategy'],
    grid_params['dataset_name']
))

# Run the quantize function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    model_name, max_new_tokens, temperature, use_beam_search, strategy, dataset_name = combination

    # Print current combination details
    print(f"Running combination {i+1}/{len(grid_combinations)}")
    print(f"  Model Name: {model_name}")
    print(f"  Max New Tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    print(f"  Use Beam Search: {use_beam_search}")
    print(f"  Strategy: {strategy}")
    print(f"  Dataset Name: {dataset_name}")

    result = run_reliability_eval(
        model_name=model_name,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_beam_search=use_beam_search,
        strategy=strategy,
        dataset_name=dataset_name,
        device=fixed_params['device'],
        seed=fixed_params['seed'],
        n_repeats=fixed_params['n_repeats'],
        n_beams=fixed_params['n_beams'],
        cache_path=CACHE_PATH
    )

    # Append result with parameter details
    results.append({
        'result': result,
        'parameters': {
            'model_name': model_name,
            'max_new_tokens': max_new_tokens,
            'temperature': temperature,
            'use_beam_search': use_beam_search,
            'strategy': strategy,
            'dataset_name': dataset_name,
            'device': fixed_params['device'],
            'n_repeats': fixed_params['n_repeats'],
            'n_beams': fixed_params['n_beams']
        }
    })

# Do something with the results
print(results)

Running combination 1/22050
  Model Name: Llama-3-8B
  Max New Tokens: 10
  Temperature: 0.1
  Use Beam Search: True
  Strategy: Fact Statement
  Dataset Name: toy-qa-dataset


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [01:08<00:00, 17.00s/it]
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.75` -- this flag is only used in sample-based generation modes. You should set `do_sample=T

  TOTAL: 1/21, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: True, PROB: 0.01, ADJ_PROB: 0.62, ENTROPY: 2.07
  TOTAL: 2/21, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: True, PROB: 0.07, ADJ_PROB: 0.77, ENTROPY: 1.67
  TOTAL: 3/21, MODEL: Llama-3-8B, QUERY: 2, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: True, PROB: 0.01, ADJ_PROB: 0.60, ENTROPY: 2.22
  TOTAL: 4/21, MODEL: Llama-3-8B, QUERY: 3, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.00, ADJ_PROB: 0.34, ENTROPY: 2.04
  TOTAL: 5/21, MODEL: Llama-3-8B, QUERY: 4, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.00, ADJ_PROB: 0.35, ENTROPY: 2.34
  TOTAL: 6/21, MODEL: Llama-3-8B, QUERY: 5, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: True, PROB: 0.00, ADJ_PROB: 0.36, ENTROPY: 2.04
  TOTAL: 7/21,

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.75` -- this flag is only used in sample-based generation modes. You should set `do_sample=T

  TOTAL: 1/100, MODEL: Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.00, ADJ_PROB: 0.50, ENTROPY: 2.32
  TOTAL: 2/100, MODEL: Llama-3-8B, QUERY: 1, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.01, ADJ_PROB: 0.64, ENTROPY: 1.17
  TOTAL: 3/100, MODEL: Llama-3-8B, QUERY: 2, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.01, ADJ_PROB: 0.63, ENTROPY: 1.55
  TOTAL: 4/100, MODEL: Llama-3-8B, QUERY: 3, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.00, ADJ_PROB: 0.49, ENTROPY: 1.96
  TOTAL: 5/100, MODEL: Llama-3-8B, QUERY: 4, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.01, ADJ_PROB: 0.64, ENTROPY: 1.87
  TOTAL: 6/100, MODEL: Llama-3-8B, QUERY: 5, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/10
    IS_CORRECT: False, PROB: 0.00, ADJ_PROB: 0.49, ENTROPY: 2.22
  TO

KeyboardInterrupt: 

In [5]:
import os
import pandas as pd

def unify_excel_files(save_dir):
    # Define paths for the unified raw table and scores table
    unified_raw_table_path = os.path.join(save_dir, "unified_raw_table.xlsx")
    unified_scores_table_path = os.path.join(save_dir, "unified_scores_table.xlsx")
    
    # Initialize empty DataFrames for unified tables
    unified_raw_table = pd.DataFrame()
    unified_scores_table = pd.DataFrame()
    
    exclude_files = ["unified_raw_table.xlsx", "unified_scores_table.xlsx"]

    # Loop through all files in the save_dir
    for filename in os.listdir(save_dir):
        if filename.endswith(".xlsx") and filename not in exclude_files:
            # Parse model, beams, max_new_tokens, temperature, and strategy from the filename
            parts = filename.replace(".xlsx", "").split('_')
            model = parts[0]
            dataset_name = parts[1]
            beam_search = parts[2]  # Either 'beam' or 'sample'
            max_new_tokens = int(parts[3])
            temperature = float(parts[5])
            strategy = '_'.join(parts[7:]).replace("raw_table", "").replace("scores", "").replace("__", "_").strip('_')

            # Load the file into a DataFrame
            file_path = os.path.join(save_dir, filename)
            df = pd.read_excel(file_path)

            # Determine if this is a raw table or a scores table
            if 'raw_table' in filename:
                # Add extra columns for model, beam_search, etc.
                df['Model'] = model
                df['Dataset Name'] = dataset_name
                df['Beam Search'] = beam_search
                df['Max New Tokens'] = max_new_tokens
                df['Temperature'] = temperature
                df['Strategy'] = strategy

                # Append to the unified raw table
                unified_raw_table = pd.concat([unified_raw_table, df], ignore_index=True)

            elif 'scores' in filename:
                # Add extra columns for model, beam_search, etc.
                df['Model'] = model
                df['Dataset Name'] = dataset_name
                df['Beam Search'] = beam_search
                df['Max New Tokens'] = max_new_tokens
                df['Temperature'] = temperature
                df['Strategy'] = strategy

                # Append to the unified scores table
                unified_scores_table = pd.concat([unified_scores_table, df], ignore_index=True)

    # Save the unified tables to Excel files
    unified_raw_table.to_excel(unified_raw_table_path, index=False)
    unified_scores_table.to_excel(unified_scores_table_path, index=False)

    print(f"Unified raw table saved to {unified_raw_table_path}")
    print(f"Unified scores table saved to {unified_scores_table_path}")

# Example usage:
unify_excel_files("results/reliability_eval")

Unified raw table saved to results/reliability_eval/unified_raw_table.xlsx
Unified scores table saved to results/reliability_eval/unified_scores_table.xlsx
